<a href="https://colab.research.google.com/github/alanchernoff/Crypto_Exchange_Vol/blob/main/ETH_Download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests -q

In [2]:
"""
Download Coinbase ETH-USD 1-minute close prices for all of 2023.
Output: eth_close_2023.csv  (rows = dates, columns = minute of day)
"""

import requests
import pandas as pd
import time
from datetime import datetime, timezone

PRODUCT_ID   = "ETH-USD"
GRANULARITY  = "ONE_MINUTE"
CANDLES_URL  = f"https://api.coinbase.com/api/v3/brokerage/market/products/{PRODUCT_ID}/candles"
MAX_CANDLES  = 300          # API hard limit per request
WINDOW_SECS  = MAX_CANDLES * 60  # 300 minutes per call

START_TS = int(datetime(2023, 1, 1, tzinfo=timezone.utc).timestamp())
END_TS   = int(datetime(2024, 1, 1, tzinfo=timezone.utc).timestamp())


def fetch_candles(start: int, end: int) -> list[dict]:
    """Fetch up to 300 1-minute candles from Coinbase Advanced Trade API."""
    params = {"start": start, "end": end, "granularity": GRANULARITY}
    for attempt in range(5):
        try:
            r = requests.get(CANDLES_URL, params=params, timeout=15)
            if r.status_code == 429:          # rate-limited
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            return r.json().get("candles", [])
        except requests.RequestException as e:
            print(f"  [retry {attempt+1}] {e}")
            time.sleep(2 ** attempt)
    return []


def download_all() -> pd.DataFrame:
    all_rows = []
    current_end = END_TS
    total_windows = (END_TS - START_TS) // WINDOW_SECS + 1
    done = 0

    print(f"Fetching ETH-USD minute candles for 2023 (~{total_windows} API calls)…")

    while current_end > START_TS:
        current_start = max(current_end - WINDOW_SECS, START_TS)

        candles = fetch_candles(current_start, current_end)
        for c in candles:
            ts = int(c["start"])
            if START_TS <= ts < END_TS:
                all_rows.append({"timestamp": ts, "close": float(c["close"])})

        current_end = current_start
        done += 1
        if done % 50 == 0:
            pct = 100 * (END_TS - current_end) / (END_TS - START_TS)
            print(f"  {pct:.1f}% complete…")

        time.sleep(0.12)   # ~8 req/s — well within rate limit

    print(f"Downloaded {len(all_rows):,} candles.")
    return pd.DataFrame(all_rows)


def pivot_to_wide(df: pd.DataFrame) -> pd.DataFrame:
    """Reshape to: rows = M/D/YYYY, columns = H:MM:SS (minute of day)."""
    df["dt"]   = pd.to_datetime(df["timestamp"], unit="s", utc=True)
    df["date"] = df["dt"].dt.strftime("%-m/%-d/%Y")   # e.g. 1/1/2023
    df["time"] = df["dt"].dt.strftime("%-H:%M:%S")    # e.g. 0:00:00

    wide = (
        df.pivot_table(index="date", columns="time", values="close", aggfunc="last")
        .reindex(columns=sorted(df["time"].unique(),
                                key=lambda t: int(t.split(":")[0]) * 3600
                                            + int(t.split(":")[1]) * 60))
    )
    # Sort rows chronologically
    wide.index = pd.to_datetime(wide.index, format="%m/%d/%Y")
    wide = wide.sort_index()
    wide.index = wide.index.strftime("%-m/%-d/%Y")
    wide.index.name = None
    return wide


if __name__ == "__main__":
    raw = download_all()

    if raw.empty:
        print("No data returned — check your internet connection or the API endpoint.")
    else:
        # Remove duplicates (overlapping windows)
        raw = raw.drop_duplicates("timestamp")

        wide = pivot_to_wide(raw)

        out_path = "eth_close_2023.csv"
        wide.to_csv(out_path)
        print(f"\nSaved → {out_path}")
        print(f"Shape: {wide.shape[0]} days × {wide.shape[1]} minutes")
        print("\nPreview (first 3 rows, first 6 cols):")
        print(wide.iloc[:3, :6].to_string())

Fetching ETH-USD minute candles for 2023 (~1753 API calls)…
  2.9% complete…
  5.7% complete…
  8.6% complete…
  11.4% complete…
  14.3% complete…
  17.1% complete…
  20.0% complete…
  22.8% complete…
  25.7% complete…
  28.5% complete…
  31.4% complete…
  34.2% complete…
  37.1% complete…
  40.0% complete…
  42.8% complete…
  45.7% complete…
  48.5% complete…
  51.4% complete…
  54.2% complete…
  57.1% complete…
  59.9% complete…
  62.8% complete…
  65.6% complete…
  68.5% complete…
  71.3% complete…
  74.2% complete…
  77.1% complete…
  79.9% complete…
  82.8% complete…
  85.6% complete…
  88.5% complete…
  91.3% complete…
  94.2% complete…
  97.0% complete…
  99.9% complete…
Downloaded 527,008 candles.

Saved → eth_close_2023.csv
Shape: 365 days × 1440 minutes

Preview (first 3 rows, first 6 cols):
time      0:00:00  0:01:00  0:02:00  0:03:00  0:04:00  0:05:00
1/1/2023  1195.27  1195.16  1194.98  1195.08  1195.60  1195.75
1/2/2023  1200.29  1200.34  1199.76  1200.00  1199.87  1200.4

In [3]:
from google.colab import files
files.download("eth_close_2023.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>